# Model Merging Techniques

## LoRA Merging

LoRA merging combines trained low-rank adapters back into the base model weights. Given a base model $W$ and LoRA adapters $BA$, the merged weight is: $W_{\text{merged}} = W + \frac{\alpha}{r} BA$, where $\alpha$ is the scaling factor and $r$ is the rank. This produces a single model without adapter overhead.

```python title="example1.py"
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model and LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125M")
model = PeftModel.from_pretrained(base_model, "path/to/lora-adapter")

# Merge LoRA into base model
merged_model = model.merge_and_unload()

# Save merged model
merged_model.save_pretrained("./merged-model")
print("LoRA merged successfully.")
```

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/llm-fine-tuning/mod-23.ipynb)

```
LoRA merged successfully.
```

## Multi-Adapter Merging (TIES)

TIES (Task-Specific Inverse Scaling) merges multiple task-specific LoRA adapters by: (1) removing redundant parameters via magnitude pruning, (2) resolving sign conflicts between adapters, and (3) scaling by task importance. This enables a single model to handle multiple tasks without catastrophic forgetting.

```python title="example2.py"
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Load base model
base_model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125M")

# Load multiple task-specific adapters
adapters = {}
for task in ["summarization", "translation", "qa"]:
    model = PeftModel.from_pretrained(base_model, f"path/to/{task}-adapter")
    adapters[task] = model.peft_config

# TIES merging: combine adapters with conflict resolution
def ties_merge(base_model, adapters, pruning_ratio=0.9):
    """Merge multiple adapters using TIES strategy"""
    merged_state = base_model.state_dict().copy()
    
    # Collect all adapter weights
    adapter_weights = {}
    for task, config in adapters.items():
        # Load adapter weights (simplified)
        adapter_weights[task] = {}
    
    # Prune redundant parameters and resolve conflicts
    for param_name in merged_state:
        if 'lora' in param_name:
            # Apply magnitude pruning
            weights = [adapter_weights[task].get(param_name, 0) for task in adapters]
            mask = torch.abs(torch.stack(weights)) > torch.quantile(
                torch.abs(torch.stack(weights)), pruning_ratio
            )
            # Average non-pruned weights
            merged_state[param_name] = torch.mean(
                torch.stack(weights) * mask.float(), dim=0
            )
    
    return merged_state

merged_weights = ties_merge(base_model, adapters)
print("TIES merge completed.")
```

> **💡 Tip:** TIES works best when adapters are trained on related tasks. For unrelated tasks, consider using separate models or mixture-of-experts approaches.

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the primary benefit of LoRA merging?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387200001" value="0">
      <span>Increases model size</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387200001" value="1">
      <span>Eliminates adapter overhead for deployment</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387200001" value="2">
      <span>Improves training speed</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387200001" value="3">
      <span>Reduces model accuracy</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What does TIES merging address when combining multiple adapters?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387200002" value="0">
      <span>Increasing model parameters</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387200002" value="1">
      <span>Reducing inference latency</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387200002" value="2">
      <span>Resolving sign conflicts and pruning redundancy</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387200002" value="3">
      <span>Improving tokenization</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>